# Python Idioms for DSA

A quick tour of the Python features that show up constantly in the rest of these notebooks — slicing, comprehensions, the data model, unpacking, and the handful of built-ins and stdlib tools that keep DSA code clean. Not a full language tutorial; just the idioms worth having fluent before diving in.

**Contents**

1. **Sequences & slicing**
2. **Comprehensions & generators**
3. **The data model** — making your objects behave like built-ins
4. **Unpacking & multiple assignment**
5. **Built-ins & stdlib for DSA**
6. **Gotchas**

## 1. Sequences & slicing

`list`, `tuple`, `str`, and `range` are all **sequences** — indexable, iterable, sliceable. Slicing `s[start:stop:step]` is the workhorse: `stop` is **exclusive**, indices can be **negative** (counting from the end), and any part can be omitted. A slice always returns a **new** object (a shallow copy, for lists).

In [1]:
xs = [0, 1, 2, 3, 4, 5]
print("xs[1:4]    :", xs[1:4])      # [1, 2, 3]   (stop is exclusive)
print("xs[-2:]    :", xs[-2:])      # [4, 5]      (negative index)
print("xs[::2]    :", xs[::2])      # [0, 2, 4]   (step)
print("xs[::-1]   :", xs[::-1])     # [5, 4, 3, 2, 1, 0]  (reverse)
print("xs[:]      :", xs[:], "(a shallow copy)")
print("'hello'[1:4]:", "hello"[1:4])


xs[1:4]    : [1, 2, 3]
xs[-2:]    : [4, 5]
xs[::2]    : [0, 2, 4]
xs[::-1]   : [5, 4, 3, 2, 1, 0]
xs[:]      : [0, 1, 2, 3, 4, 5] (a shallow copy)
'hello'[1:4]: ell


## 2. Comprehensions & generators

A **comprehension** builds a list / set / dict in one readable expression — clearer and faster than an append loop. Swap the brackets for parentheses and you get a **generator**: lazy, one item at a time, O(1) memory — perfect for streaming or feeding `sum` / `any` / `max`.

In [2]:
squares = [x * x for x in range(6)]
evens   = [x for x in range(10) if x % 2 == 0]      # with a filter
pairs   = {c: i for i, c in enumerate("abc")}       # dict comprehension
mods    = {x % 3 for x in range(10)}                # set comprehension
print("list  :", squares)
print("filter:", evens)
print("dict  :", pairs)
print("set   :", mods)

gen = (x * x for x in range(10**9))                 # lazy: builds nothing yet
print("first 3 lazily   :", [next(gen) for _ in range(3)])
print("sum of a genexpr :", sum(x * x for x in range(6)))   # no intermediate list


list  : [0, 1, 4, 9, 16, 25]
filter: [0, 2, 4, 6, 8]
dict  : {'a': 0, 'b': 1, 'c': 2}
set   : {0, 1, 2}
first 3 lazily   : [0, 1, 4]
sum of a genexpr : 55


## 3. The data model — making your objects behave like built-ins

Python's **dunder** (double-underscore) methods are hooks the language calls for you. Implement them and your own classes work with `len()`, `[]`, `in`, `for`, `==`, `<`, dict keys, and more. The DSA notebooks lean on these everywhere (a `Stack` with `__len__`, a `Node` with `__slots__`, a tree with `__contains__`):

In [3]:
class Ring:
    """A tiny sequence, to show the protocols."""
    def __init__(self, *items):
        self._items = list(items)
    def __len__(self):              # enables len(r)
        return len(self._items)
    def __getitem__(self, i):       # enables r[i], iteration, and `in`
        return self._items[i]
    def __repr__(self):             # what you see when you print it
        return f"Ring({self._items})"

r = Ring(10, 20, 30)
print("len       :", len(r))
print("index     :", r[1])
print("iterate   :", [x for x in r])    # works via __getitem__
print("membership:", 20 in r)
print("repr      :", r)


len       : 3
index     : 20
iterate   : [10, 20, 30]
membership: True
repr      : Ring([10, 20, 30])


The key contracts to remember: `__eq__` + `__hash__` together make instances usable as **dict keys / set members** (equal objects must hash equal — the dict & set notebook); `__lt__` makes them **sortable and heap-able**. Define them consistently or you'll get subtle bugs.

## 4. Unpacking & multiple assignment

Python assigns several names at once, which makes swaps and destructuring clean — you'll see these all over the algorithm code:

In [4]:
a, b = 1, 2
a, b = b, a                         # swap, no temp variable
print("swap      :", a, b)

first, *rest = [10, 20, 30, 40]     # star captures the remainder
print("first/rest:", first, rest)
*init, last = [10, 20, 30, 40]
print("init/last :", init, last)

x, y = (3, 4)                       # destructure a tuple
print("x, y      :", x, y)

for i, value in enumerate(["a", "b"]):          # index + value together
    print("  enumerate:", i, value)
for name, age in zip(["Ada", "Alan"], [36, 41]):  # iterate in lockstep
    print("  zip      :", name, age)


swap      : 2 1
first/rest: 10 [20, 30, 40]
init/last : [10, 20, 30] 40
x, y      : 3 4
  enumerate: 0 a
  enumerate: 1 b
  zip      : Ada 36
  zip      : Alan 41


## 5. Built-ins & stdlib for DSA

A small toolkit covers most needs — prefer these over hand-rolling. Each maps onto a notebook in this repo:

In [5]:
nums = [5, 3, 8, 1]
print("sorted (key)   :", sorted(["aa", "b", "ccc"], key=len))   # by length
print("max / min      :", max(nums), min(nums))
print("sum / any / all:", sum(nums), any(n > 7 for n in nums), all(n > 0 for n in nums))

from collections import deque, Counter, defaultdict
print("deque          :", deque([1, 2, 3]))                      # O(1) at both ends
print("Counter top-2  :", Counter("mississippi").most_common(2))
dd = defaultdict(list); dd["x"].append(1)
print("defaultdict    :", dict(dd))

import heapq, bisect
print("heapq.nsmallest:", heapq.nsmallest(2, nums))              # partial sort
print("bisect spot    :", bisect.bisect_left([1, 3, 5, 7], 4))   # binary search


sorted (key)   : ['b', 'aa', 'ccc']
max / min      : 8 1
sum / any / all: 17 True True
deque          : deque([1, 2, 3])
Counter top-2  : [('i', 4), ('s', 4)]
defaultdict    : {'x': [1]}
heapq.nsmallest: [1, 3]
bisect spot    : 2


These map straight onto the notebooks: `collections` / `heapq` / `bisect` each get a full treatment; `sorted(key=...)` is the sorting notebook; `Counter` / `defaultdict` live in the dict & set notebook.

## 6. Gotchas

Two Python traps that bite DSA code specifically:

In [6]:
# 1) MUTABLE DEFAULT ARGUMENT - the default is created ONCE and shared across calls!
def buggy(x, acc=[]):
    acc.append(x)
    return acc
print("buggy(1), buggy(2):", buggy(1), buggy(2))   # BOTH show [1, 2] - same list!

def fixed(x, acc=None):
    if acc is None:
        acc = []                    # a fresh list per call
    acc.append(x)
    return acc
print("fixed(1), fixed(2):", fixed(1), fixed(2))   # [1] then [2]

# 2) lists/dicts/sets are mutable and shared BY REFERENCE (ints/strs/tuples aren't)
a = [1, 2, 3]
b = a                               # NOT a copy - the same object
b.append(4)
print("aliasing  :", a, "| b is a:", b is a)
c = a[:]                            # shallow copy - independent at the top level
c.append(5)
print("after copy:", a, c)


buggy(1), buggy(2): [1, 2] [1, 2]
fixed(1), fixed(2): [1] [2]
aliasing  : [1, 2, 3, 4] | b is a: True
after copy: [1, 2, 3, 4] [1, 2, 3, 4, 5]


The mutable-default and aliasing traps come straight from the arrays notebook's `[[0]*n]*m` discussion. When in doubt about whether two names share an object, check with `is` or `id()`. (And the shallow-copy caveat: `a[:]` copies the outer list but not nested objects — use `copy.deepcopy` for that.)